In [0]:
-- EDA (EXPLORATORY DATA ANALYSIS) ON BRIGHT_TV
-- USER PROFILES
-- VIEWERSHIP


-- VIEW WHOLE TABLE BEFORE ANALYSIS
SELECT *
FROM bright.bright_tv.bright_tv_dataset_user_profiles
LIMIT 10;


-- HOW BIG IS THE DATA
SELECT COUNT(*) AS number_of_rows 
FROM bright.bright_tv.bright_tv_dataset_user_profiles;


--- WANT TO CHECK FOR DUPLICATES- Check for userids.
--- If the number of rows is equal to the number of UserIDs, then there are no duplicates in the data.

SELECT COUNT(*) AS number_of_rows,
       COUNT(DISTINCT UserID) AS number_of_subs
FROM bright.bright_tv.bright_tv_dataset_user_profiles; -- number of rows=number of unique userids. So, no duplicates.

-- OR
SELECT UserID, COUNT(*) AS duplicate_cnt 
FROM bright.bright_tv.bright_tv_dataset_user_profiles
Group by UserID
Having COUNT(*) > 1; -- no rows returned, therefore, no duplicates.

--- Any rows where userID is null

SELECT COUNT(*) 
FROM bright.bright_tv.bright_tv_dataset_user_profiles
WHERE UserID IS NULL; --- no rows, the count is zero.


--- Gender analysis
SELECT DISTINCT gender
FROM bright.bright_tv.bright_tv_dataset_user_profiles; -- male, female, none, empty space

SELECT COUNT(*)
FROM bright.bright_tv.bright_tv_dataset_user_profiles
WHERE gender IS NULL; -- zero

SELECT COUNT(*)
FROM bright.bright_tv.bright_tv_dataset_user_profiles
WHERE gender=' '; -- 218 rows with an empty space

SELECT COUNT(*) AS Cnt,
       COUNT(DISTINCT UserID) AS subs,
    CASE 
        WHEN gender = ' ' THEN 'Unknown'
        When gender ilike 'None' THEN 'Unknown'
        ELSE gender
        END AS gender ---maintains the other genders that are not mentioned in the case statement the same.
FROM bright.bright_tv.bright_tv_dataset_user_profiles
GROUP BY gender; -- because of aggregate function 'count()'
-- the output table us ' '= Unknown and 'none'=unknown
-- but the visualization/barchart combines the none and the empty space, so we are sorted.


--- TRIM FUNCTION FROM CHATGPT
SELECT
    COUNT(DISTINCT UserID) AS subs,
    gender
FROM (
    SELECT
        UserID,
        CASE
            WHEN gender IS NULL THEN 'Unknown'
            WHEN TRIM(LOWER(gender)) IN ('', 'none') THEN 'Unknown'
            ELSE gender
        END AS gender
    FROM bright.bright_tv.bright_tv_dataset_user_profiles
) t
GROUP BY gender;


--- Race analysis

SELECT Distinct Race
FROM bright.bright_tv.bright_tv_dataset_user_profiles; --- there is 'other','none' and an empty space which needs to be solved/or combined.

SELECT DISTINCT--- returns all the different races and removes duplicates
    CASE
        WHEN Race IN ('other') THEN 'None'
        WHEN Race =' ' THEN 'None'
    ELSE Race -- return the other races the same
    End as Race
FROM bright.bright_tv.bright_tv_dataset_user_profiles; -- sorted.

   
--- Is there any row where the RACE ccolumn is NULL?

SELECT COUNT(*) -- counts all the rows in the RACE column
from bright.bright_tv.bright_tv_dataset_user_profiles
WHERE Race IS NULL; -- Zero which is good.


--- Province analysis

SELECT DISTINCT Province
FROM bright.bright_tv.bright_tv_dataset_user_profiles; --- There is a 'none' and an empty space that needs to be sorted.
-- We need to group them into 1 row/category.

SELECT DISTINCT
    CASE
        WHEN Province IN (' ') THEN 'Uncategorized'
        WHEN Province IN('None') THEN 'Uncategorized'
        ELSE Province
        END AS region
FROM bright.bright_tv.bright_tv_dataset_user_profiles; --- Sorted.

---- Age Analysis

SELECT  Min(Age) as min_age, -- = 0
        max(Age) AS max_age  -- = 114
FROM bright.bright_tv.bright_tv_dataset_user_profiles;

---Is there a row where age is NULL?
SELECT COUNT(*)
FROM bright.bright_tv.bright_tv_dataset_user_profiles
WHERE Age IS NULL; --- ALL rows have age values. Sorted.

---- Age brackets/grouping

SELECT COUNT(DISTINCT userid) AS subs, 
    CASE
        WHEN age = 0 THEN 'Infants'
        WHEN age BETWEEN 1 AND 12 THEN 'Kids'
        WHEN age BETWEEN 13 AND 19 THEN 'Teens'
        WHEN age BETWEEN 20 AND 35 THEN 'Youth'
        WHEN age BETWEEN 36 AND 50 THEN 'Adults'
        WHEN age BETWEEN 51 AND 65 THEN 'Senior'
        ELSE 'Madala' --- Even if you don't use 'ELSE',The outcome will be the same.
                    --- Another alternative is: WHEN age>65 the 'Madala'
                    --- will yield the same outcome
    END AS age_group ---Group by the age group otherwise you will have 71 rows if you group as 'age'
FROM bright.bright_tv.bright_tv_dataset_user_profiles
GROUP BY age_group;

----- NOW TO COMBINE THE ANALYSIS, USE CTE:Combining all the case statements under 1 SELECT STATEMENT.
---------------------------------------------------------------------------------------------------------------
-- CTE: Is a way to break down code that is coming from separate tables.
-- Process code on one table before joining it with the 2nd code from a different table.
-- '<>' and '!=' is the same thing= not equal to


WITH user_profiles AS (
SELECT UserID, -- primary key but it is a foreign key on the viewership table.

    CASE 
        WHEN gender = ' ' THEN 'Unknown'
        When gender ilike 'None' THEN 'Unknown'
        ELSE gender
        END AS gender,

    CASE
        WHEN Race IN ('other') THEN 'None'
        WHEN Race =' ' THEN 'None'
    ELSE Race -- return the other races the same
    End as Race,

    CASE
        WHEN Province IN (' ') THEN 'Uncategorized'
        WHEN Province IN('None') THEN 'Uncategorized'
        ELSE Province
        END AS region,
    
    CASE
        WHEN age = 0 THEN 'Infants'
        WHEN age BETWEEN 1 AND 12 THEN 'Kids'
        WHEN age BETWEEN 13 AND 19 THEN 'Teens'
        WHEN age BETWEEN 20 AND 35 THEN 'Youth'
        WHEN age BETWEEN 36 AND 50 THEN 'Adults'
        WHEN age BETWEEN 51 AND 65 THEN 'Senior'
        ELSE 'Madala'
    END AS age_group,

    CASE
        WHEN (`Social Media Handle` IS NOT NULL) OR (`Social Media Handle`!=' ') OR (`Social Media Handle` NOT IN ('None') ) THEN 1
        ELSE 0
    END AS social_flag,

    CASE 
        WHEN (Email IS NOT NULL) OR (Email <>' ') OR (Email NOT IN ('None') ) THEN 1
        ELSE 0
    END AS email_flag

FROM bright.bright_tv.bright_tv_dataset_user_profiles
),
viewership AS
(
    SELECT

    -- Date
    COALESCE(UserID0,userid4,0)AS userid, --- Initially, I named this column as 'user_id' and when I had to join the 2 tables, the code failed because of the ambiguity. Now Left joining the two tables works. So, I am SORTED!!!!
    
    TO_CHAR(TO_DATE(RecordDate2),'yyyyMM') AS month_id, -- TO_CHAR(): converts a date into a string and TO_DATE(): converts a string into a date
    TO_DATE(RecordDate2) AS watch_date, -- extract the date from the time stamp
   -- TIME(RecordDate2) AS watch_time,
    DAY(RecordDate2) AS day_of_month,
    DAYNAME(RecordDate2) AS day_name,
    MONTHNAME(RecordDate2) AS month_name,

--TIME
-- A timestamp records the date and time of an event

date_format(RecordDate2,'HH:mm:ss') AS watch_time, -- The DATE_FORMAT(): Extracts the date and or time from the timestamp
HOUR(RecordDate2) AS hour_of_day,

CASE 
    WHEN watch_time BETWEEN '00:00:00' AND '05:59:59' THEN 'Night_surfer'
    WHEN watch_time BETWEEN '06:00:00' AND '11:59:59' THEN 'Morning'
    WHEN watch_time BETWEEN '12:00:00' AND '16:59:59' THEN 'Afternoon'
    WHEN watch_time BETWEEN '17:00:00' AND '29:59:59' THEN 'Evening'
END AS time_of_day,

date_format(`Duration 2`,'HH:mm:ss') AS duration, 
CASE 
    WHEN duration BETWEEN '00:00:00' AND '00:30:00' THEN 'low_stream'
    WHEN duration BETWEEN '00:30:01' AND '00:59:59' THEN 'med_stream'
    WHEN duration> '01:00:00' THEN 'high_stream'
END AS stream_duration,


CASE 
    WHEN day_name IN ('Sat','Sun') THEN 'weekend'
    ELSE 'weekday'
END AS day_category,


CASE
    WHEN Channel2 IN ('SawSee','SawSee') THEN 'Sawsee'  
    WHEN Channel2 IN('SuperSport Live Events', 'Live on SuperSport','Supersport Live Events','DStv Events 1') THEN 'Live Events'
       
    ELSE Channel2
    END AS TV_Channel


FROM bright.bright_tv.bright_tv_dataset_viewership
)

SELECT COALESCE(A.userid,B.userid) AS sub_id,
       month_id,
       watch_date,
       day_of_month,
       day_name,
       month_name,
       watch_time,
       hour_of_day,
       time_of_day,
       duration,
       stream_duration,
       day_category,
       TV_Channel,
       gender,
       Race,
       region,
       age_group,
       social_flag,
       email_flag

FROM viewership AS A
LEFT JOIN
user_profiles AS B
on A.userid=B.userid
GROUP BY ALL
;

--- LEFT JOIN FINALLY WORKED!
-- The issue was with the ambiguity of the 'Userid/ userID,user_id'
-- After running the CTE,
--You need to aggregate, by counting and grouping by all.
-- Download the outcome table as an excel file (csv)
-- start building visuals from this aggregated table.

-- Subscribers per region: USE COUNT DISTINCT()
-- Views per region: USE COUNT()




Databricks visualization. Run in Databricks to view.